# TUGAS MANDIRI — Pertemuan 4
## Migrasi Analisis Data Transaksi E-Commerce dari pandas ke PySpark

| | |
|---|---|
| **Nama** | Hilmi Mufid |
| **NIM** | 2505060046 |
| **Program Studi** | S1 Teknologi Informasi |
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 4 |

---

### Konteks

Tim engineering platform e-commerce (skenario yang sama dari Pertemuan 2-3) resmi meminta seluruh proses analisis data yang sebelumnya memakai pandas **dipindahkan ke PySpark**, karena volume data transaksi diperkirakan akan tumbuh sangat besar dalam waktu dekat sehingga pandas (yang memuat semua data ke RAM) tidak lagi memadai.

Notebook ini membuktikan bahwa seluruh alur analisis dapat direplikasi menggunakan **PySpark**, dengan data **dibaca langsung dari HDFS** — bukan dari berkas lokal, dan **tanpa menggunakan pandas** sama sekali di bagian analisis (A-E).

> **Catatan lingkungan & path:** Dijalankan di laptop Ubuntu (dual-boot). Mengikuti konvensi direktori kerja pribadi yang sudah dipakai sejak Pertemuan 3 (`/home/mufid/praktikum-bigdata0046`), maka `/user/mahasiswa/tugas4` pada instruksi soal disesuaikan menjadi `/home/mufid/praktikum-bigdata0046/tugas4`.


---
## Persiapan: Membuat Dataset dan Upload ke HDFS

Jalankan cell berikut untuk membuat dataset transaksi bulan September 2026 dan mengunggahnya ke HDFS. (Disalin dari modul Pertemuan 4, path HDFS disesuaikan dengan konvensi folder pribadi.)


In [1]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS (path disesuaikan dengan folder kerja pribadi)
!hdfs dfs -mkdir -p /home/mufid/praktikum-bigdata0046/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /home/mufid/praktikum-bigdata0046/tugas4/
print("Berhasil diunggah ke HDFS: /home/mufid/praktikum-bigdata0046/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /home/mufid/praktikum-bigdata0046/tugas4/transaksi_september_2026.csv


### Membuat SparkSession

Sesuai instruksi, seluruh pekerjaan A-E wajib menggunakan PySpark, bukan pandas, dan data wajib dibaca langsung dari HDFS.


In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, avg, when

spark = SparkSession.builder \
    .appName("Tugas4-MigrasiPySpark") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


---
## A. Membaca dan Eksplorasi Awal *(bobot 15%)*

**Penjelasan:** Dataset dibaca **langsung dari HDFS** menggunakan `spark.read.csv()` dengan prefix `hdfs://localhost:9000/...` — bukan dari path lokal. Opsi `header=True` memberi tahu Spark bahwa baris pertama adalah nama kolom, dan `inferSchema=True` membuat Spark otomatis menebak tipe data tiap kolom (integer, string, double, dsb).


In [13]:
HDFS_PATH = "hdfs://localhost:9000/home/mufid/praktikum-bigdata0046/tugas4/transaksi_september_2026.csv"

df = spark.read.csv(HDFS_PATH, header=True, inferSchema=True)

print("Struktur skema DataFrame:")
df.printSchema()

print("Jumlah baris:", df.count())

print("10 baris pertama:")
df.show(10)

Struktur skema DataFrame:
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
10 baris pertama:
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00

---
## B. Menangani Data Kosong *(bobot 15%)*

**Penjelasan:** Kolom `rating` sengaja dibuat memiliki nilai kosong (`NaN`) pada tahap pembuatan dataset. Sebelum data kosong ditangani, pertama-tama dihitung berapa banyak baris yang kosong pada kolom tersebut.

**Pilihan yang digunakan: `df.na.fill()`, bukan `df.na.drop()`.**

**Alasan:** Rating adalah opini subjektif pembeli yang sifatnya opsional — banyak pembeli memang tidak memberi rating meskipun transaksinya tetap sah dan valid. Jika baris dengan rating kosong dibuang (`na.drop()`), maka baris data transaksi yang sebenarnya valid dan mengandung informasi penting lain (kota, kategori, total pendapatan, metode pembayaran) ikut hilang seluruhnya, sehingga total pendapatan dan jumlah transaksi pada analisis selanjutnya (bagian D) akan under-counted / tidak akurat. Karena itu, nilai kosong pada `rating` lebih tepat diisi (`fill`) dengan nilai netral (angka `0`, menandakan "tidak ada rating") daripada membuang keseluruhan barisnya.


In [4]:
# Menghitung jumlah baris dengan rating kosong
jumlah_rating_kosong = df.filter(col("rating").isNull()).count()
print(f"Jumlah baris dengan rating kosong: {jumlah_rating_kosong}")

# Menangani data kosong: mengisi (fill) rating yang kosong dengan 0
# (0 dipilih sebagai penanda "tidak ada rating", bukan skor rating sungguhan 1-5)
df = df.na.fill({"rating": 0})

# Verifikasi tidak ada lagi nilai kosong pada kolom rating
jumlah_rating_kosong_setelah = df.filter(col("rating").isNull()).count()
print(f"Jumlah baris dengan rating kosong setelah ditangani: {jumlah_rating_kosong_setelah}")

df.select("order_id", "rating").show(10)

Jumlah baris dengan rating kosong: 204
Jumlah baris dengan rating kosong setelah ditangani: 0
+--------+------+
|order_id|rating|
+--------+------+
|ORD-3000|   4.0|
|ORD-3001|   5.0|
|ORD-3002|   3.0|
|ORD-3003|   4.0|
|ORD-3004|   4.0|
|ORD-3005|   4.0|
|ORD-3006|   5.0|
|ORD-3007|   0.0|
|ORD-3008|   5.0|
|ORD-3009|   3.0|
+--------+------+
only showing top 10 rows



---
## C. Transformasi Data *(bobot 20%)*

**Penjelasan:** Dua kolom baru ditambahkan menggunakan `withColumn()`:
1. `total_pendapatan` = `unit_terjual x harga_satuan`.
2. `tier_transaksi` — kolom kategorikal yang dihitung dengan fungsi `when().otherwise()` dari `pyspark.sql.functions`, setara dengan `if-else` bertingkat di pandas, bernilai `"Besar"` jika `total_pendapatan > 500000`, atau `"Kecil"` jika sebaliknya.


In [5]:
# Menambahkan kolom total_pendapatan
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# Menambahkan kolom tier_transaksi berdasarkan ambang batas total_pendapatan
df = df.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

df.select("order_id", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



---
## D. Analisis dengan GroupBy *(bobot 30%)*

**Penjelasan:** Ketiga pertanyaan dijawab murni dengan sintaks PySpark (`groupBy`, `agg`, `orderBy`), tanpa pandas sama sekali.


### D.1 — Kategori dengan `total_pendapatan` tertinggi

In [6]:
kategori_tertinggi = df.groupBy("kategori").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")
).orderBy(col("total_pendapatan").desc())

kategori_tertinggi.show()

print("Jawaban: kategori dengan total pendapatan tertinggi adalah baris paling atas pada tabel di atas.")

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+

Jawaban: kategori dengan total pendapatan tertinggi adalah baris paling atas pada tabel di atas.


### D.2 — Kota dengan jumlah transaksi tier "Besar" terbanyak

In [7]:
kota_tier_besar = df.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .agg(count("order_id").alias("jumlah_transaksi_besar")) \
    .orderBy(col("jumlah_transaksi_besar").desc())

kota_tier_besar.show()

print("Jawaban: kota dengan transaksi tier Besar terbanyak adalah baris paling atas pada tabel di atas.")

+----------+----------------------+
|      kota|jumlah_transaksi_besar|
+----------+----------------------+
|      Solo|                    92|
|  Magelang|                    78|
|   Kebumen|                    78|
|Yogyakarta|                    75|
| Purworejo|                    66|
|  Semarang|                    65|
+----------+----------------------+

Jawaban: kota dengan transaksi tier Besar terbanyak adalah baris paling atas pada tabel di atas.


### D.3 — Rata-rata `rating` per `metode_pembayaran`

In [8]:
rating_per_metode = df.groupBy("metode_pembayaran").agg(
    avg("rating").alias("rata_rata_rating")
).orderBy(col("rata_rata_rating").desc())

rating_per_metode.show()

+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|         E-Wallet|             3.292|
|     Kartu Kredit|3.1910569105691056|
+-----------------+------------------+



---
## E. Menyimpan Hasil ke HDFS *(bobot 20%)*

**Penjelasan:** DataFrame hasil olahan bagian C (lengkap dengan kolom `total_pendapatan` dan `tier_transaksi`) disimpan sebagai CSV baru ke HDFS menggunakan `df.write.csv()`, kemudian diverifikasi dengan `hdfs dfs -ls`.


In [9]:
OUTPUT_PATH = "hdfs://localhost:9000/home/mufid/praktikum-bigdata0046/tugas4/hasil_olahan_september"

# Menyimpan DataFrame hasil olahan ke HDFS dalam format CSV
# mode("overwrite") memastikan tidak error jika folder sudah pernah ada sebelumnya
df.write.mode("overwrite").csv(OUTPUT_PATH, header=True)

print("Data berhasil disimpan ke HDFS.")

Data berhasil disimpan ke HDFS.


In [10]:
# Verifikasi: menampilkan isi folder hasil di HDFS
!hdfs dfs -ls /home/mufid/praktikum-bigdata0046/tugas4/hasil_olahan_september

Found 2 items
-rw-r--r--   3 mufid supergroup          0 2026-09-16 21:39 /home/mufid/praktikum-bigdata0046/tugas4/hasil_olahan_september/_SUCCESS
-rw-r--r--   3 mufid supergroup      97296 2026-09-16 21:39 /home/mufid/praktikum-bigdata0046/tugas4/hasil_olahan_september/part-00000-c1872090-e8ed-4dbd-af85-543a2bfebc21-c000.csv


**Mengapa hasilnya berupa beberapa berkas partisi (`part-00000...`, dst.), bukan satu berkas tunggal seperti pandas?**

Spark bekerja secara **terdistribusi**: data yang diproses dibagi menjadi beberapa *partition* yang dapat diolah secara paralel oleh beberapa *executor* sekaligus (pada praktikum ini disimulasikan lewat banyak thread di mode `local[*]`, memanfaatkan seluruh core CPU laptop). Setiap partisi menulis hasilnya sendiri-sendiri ke berkas terpisah (`part-00000`, `part-00001`, dst.) secara bersamaan, tanpa perlu saling menunggu atau digabung jadi satu berkas dulu. Pendekatan ini jauh lebih efisien untuk data berskala besar dibanding pandas yang menulis satu berkas tunggal secara sekuensial dari satu proses saja — karena jika dipaksa menjadi satu berkas, seluruh data harus dikumpulkan ke satu node terlebih dahulu (`coalesce(1)` atau `repartition(1)`), yang justru menghilangkan keuntungan pemrosesan paralel dan bisa menyebabkan *out of memory* pada data yang sangat besar.


---
## Menutup SparkSession


In [11]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
